## Data Loader

In [1]:
from langchain_core.documents import Document
import re

def clean_text(text):
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove special characters (optional, depending on requirements)
    # text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    return text

reviews_raw = [
    """No issues till now\nYou can buy it for students daily use or for your parents who are started to use\nTotally worth on this price\n6 people found this helpful""",
    """According to price , it is the best . Screen is large , speakers are loud , long lasting battery,light weight and usable camera. Best for giving to grandparents who want to learn using smartphone""",
    """This is a good entry level phone with good large 6.7 in display. Large battery and android 16 with one ui 8. A great phone for normal use. Don’t expect much from camera and gaming performance as it is an entry level phone.""",
    """This is marketed as a budget phone. Nonetheless, it has the quality and feel of a midrange product. It does all the essentials well, including Internet browsing, WhatsApp, etc. The only slight issue has been dropped wi-fi connection despite close proximity to the router. It'll switch to mobile data connection and this could consume a lot of data if you don't realise. You can change connection settings to avoid rapid switching from WiFi to data.""",
    """This phone has been malfunctioning from the beginning even though there is no physical damage. The phone often switches off by itself, which makes it very difficult to use.\nI am also facing connectivity problems. Both Wi-Fi and Bluetooth do not connect.\n\nI also tried to replace the product since there is a 10-day replacement option mentioned. However, my experience with customer care has been very poor. Whenever I contact them, I do not receive proper help or support.\nI would not recommend this phone or buying any electronics from amazon ever""",
    """Samsung Galaxy M07 Is Good Phone But You Can't Do Heavy Gaming like 1gb plus games in This Phone If You Try To Play Heavy The Started To Hang And You Can't play at that moment.Otherwise phone is Good And Camera Is Also Good""",
    """Biggest problem with the phone is, if you go out of network, it won't auto register until you move it to flight mode and remove once"""
]

# Convert raw reviews to Document objects with metadata
cleaned_reviews_raw = [clean_text(review) for review in reviews_raw]
reviews = [Document(page_content=review, metadata={"source": f"review_{i+1}"}) for i, review in enumerate(cleaned_reviews_raw)]

print(f"Loaded {len(reviews)} reviews as Document objects after cleaning.")

Loaded 7 reviews as Document objects after cleaning.


## Real Data Collection from Various Platforms

### Amazon and Flipkart Reviews

For Amazon and Flipkart, you would typically use web scraping libraries (like Beautiful Soup or Scrapy) or leverage available APIs (if permitted and accessible) to extract product reviews. Below is a placeholder for how you would collect data and integrate it with your existing `reviews` list.

In [2]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from bs4 import BeautifulSoup
import re

# Existing review list
reviews = []

# Simple cleaning function
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# URLs
urls = [
    "https://www.amazon.in/product-reviews/B0FN7QTRPY/",
    "https://www.flipkart.com/samsung-galaxy-m07-black-64-gb/product-reviews/itm0b84fbcd25ac9"
]

# Load webpages
loader = WebBaseLoader(urls)
web_docs = loader.load()

# Extract review-like text
amazon_flipkart_raw = []

for doc in web_docs:
    soup = BeautifulSoup(doc.page_content, "html.parser")

    # Extract visible text
    text = soup.get_text(separator=" ", strip=True)

    # Split into rough sentences
    chunks = text.split(". ")

    # Keep meaningful chunks
    for chunk in chunks:
        if len(chunk.split()) > 6:
            amazon_flipkart_raw.append(chunk)

# Limit sample size
amazon_flipkart_raw = amazon_flipkart_raw[:10]

# Clean reviews
cleaned_reviews = [clean_text(review) for review in amazon_flipkart_raw]

# Convert to Document objects
new_reviews = [
    Document(
        page_content=review,
        metadata={"source": f"amazon_flipkart_{i+1}"}
    )
    for i, review in enumerate(cleaned_reviews)
]

# Extend main list
reviews.extend(new_reviews)

print(f"Added {len(new_reviews)} reviews")
print(f"Total reviews: {len(reviews)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Added 7 reviews
Total reviews: 7


### YouTube Comments

For YouTube comments, you would typically use the YouTube Data API to retrieve comments from relevant product review videos. You'll need an API key for this, and you should be mindful of API quotas. You can search for videos by keywords (e.g., "phone model review") and then fetch comments for those videos.

In [3]:
import requests

response = requests.get("https://www.youtube.com")

print(response.status_code)

200


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
from deep_translator import GoogleTranslator
from langchain_core.documents import Document

reviews = []

video_ids = [
    "tmoOJIUEH9k",
    "HazHU_WkErY"
]

api = YouTubeTranscriptApi()

for video_id in video_ids:

    try:
        transcript_list = api.list(video_id)

        try:
            # Try English first
            transcript = transcript_list.find_transcript(['en'])

            fetched = transcript.fetch()

            full_text = " ".join(
                [entry.text for entry in fetched]
            )

            detected_language = "en"

        except:

            # Use first available transcript
            transcript = next(iter(transcript_list))

            fetched = transcript.fetch()

            original_text = " ".join(
                [entry.text for entry in fetched]
            )

            detected_language = transcript.language_code

            # Translate externally
            full_text = GoogleTranslator(
                source='auto',
                target='en'
            ).translate(original_text)

            print(f"Translated {detected_language} -> en")

        reviews.append(
            Document(
                page_content=full_text,
                metadata={
                    "source": f"youtube_{video_id}",
                    "language": detected_language
                }
            )
        )

        print(f"Loaded: {video_id}")

    except Exception as e:
        print(f"Failed: {video_id}")
        print(e)

print(f"Total reviews: {len(reviews)}")

Failed: tmoOJIUEH9k
Request exception can happen due to an api connection error. Please check your connection and try again
Loaded: HazHU_WkErY
Total reviews: 1


After adding new data, you might want to re-run the sentiment analysis and issue extraction steps to update the metadata for the newly added documents, and then re-initialize your `splitter`, `vector_store`, and `retrievers` to include the expanded dataset.

### Important: Re-run Downstream Cells

Since the data loading methods have been updated, please **re-run all cells from the 'Sentiment Analysis and Issue Extraction' section onwards** to ensure that the `reviews` list, sentiment analysis, issue extraction, chunking, vector store, retrievers, and RAG chain are all updated with the potentially new data from the loaders.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

In [7]:
chunks = splitter.split_documents(reviews)
print(f"Split into {len(chunks)} chunks.")

Split into 7 chunks.


In [8]:
from transformers import pipeline

# Initialize a sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis")

# Perform sentiment analysis and update document metadata
for doc in reviews:
    # Analyze sentiment of the page_content
    result = sentiment_analyzer(doc.page_content)[0]
    sentiment_label = result['label'] # e.g., 'POSITIVE', 'NEGATIVE', 'NEUTRAL'
    sentiment_score = result['score']

    # Add sentiment to the document's metadata
    doc.metadata['sentiment_label'] = sentiment_label
    doc.metadata['sentiment_score'] = sentiment_score

print(f"Added sentiment analysis to {len(reviews)} reviews.")
# Display the first document with its new metadata for verification
print("\nFirst document with sentiment metadata:")
print(reviews[0])

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (693 > 512). Running this sequence through the model will result in indexing errors


RuntimeError: The size of tensor a (693) must match the size of tensor b (512) at non-singleton dimension 1

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field
from typing import List

# Define Pydantic model for structured issue extraction
class ExtractedIssues(BaseModel):
    issues: List[str] = Field(description="List of negative issues or problems identified in the phone review.")

# Set up the parser for issue extraction
issue_parser = PydanticOutputParser(pydantic_object=ExtractedIssues)

# Define the prompt template for issue extraction
issue_extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract only the negative issues or problems mentioned in the following phone review and return them as a list of strings in JSON format. If no issues are mentioned, return an empty list. {format_instructions}\nReview: {review}"),
    ("user", "Extract issues from the above review."),
])

# Create an LLM chain for issue extraction
issue_extraction_chain = (
    {
        "review": RunnablePassthrough(),
        "format_instructions": lambda x: issue_parser.get_format_instructions()
    }
    | issue_extraction_prompt
    | llm # Use the already initialized 'llm' (ChatGroq)
    | issue_parser
)

# Perform issue extraction and update document metadata
for i, doc in enumerate(reviews):
    try:
        extracted_issues = issue_extraction_chain.invoke(doc.page_content)
        # Only add 'extracted_issues' to metadata if the list is not empty
        if extracted_issues.issues:
            doc.metadata['extracted_issues'] = extracted_issues.issues
        else:
            # If no issues are extracted, ensure the key is not present in metadata
            if 'extracted_issues' in doc.metadata:
                del doc.metadata['extracted_issues']
    except Exception as e:
        print(f"Error extracting issues for review {i+1}: {e}")
        # In case of error, ensure the key is not present
        if 'extracted_issues' in doc.metadata:
            del doc.metadata['extracted_issues']

print(f"Added issue extraction to {len(reviews)} reviews.")
# Display the first document with its new metadata for verification
print("\nFirst document with sentiment and issue metadata:")
print(reviews[0])
print("\nSecond document (likely negative) with sentiment and issue metadata:")
print(reviews[3]) # This review was marked as NEGATIVE earlier
print("\nFifth document (likely problematic) with sentiment and issue metadata:")
print(reviews[4]) # This review was also marked as NEGATIVE earlier

Added issue extraction to 23 reviews.

First document with sentiment and issue metadata:
page_content='No issues till now You can buy it for students daily use or for your parents who are started to use Totally worth on this price 6 people found this helpful' metadata={'source': 'review_1', 'sentiment_label': 'POSITIVE', 'sentiment_score': 0.9978659749031067}

Second document (likely negative) with sentiment and issue metadata:
page_content='This is marketed as a budget phone. Nonetheless, it has the quality and feel of a midrange product. It does all the essentials well, including Internet browsing, WhatsApp, etc. The only slight issue has been dropped wi-fi connection despite close proximity to the router. It'll switch to mobile data connection and this could consume a lot of data if you don't realise. You can change connection settings to avoid rapid switching from WiFi to data.' metadata={'source': 'review_4', 'sentiment_label': 'NEGATIVE', 'sentiment_score': 0.9289306402206421, 'e

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
import os
from getpass import getpass

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Please provide your Google API key: ")

embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

### Initialize ChromaDB and Store Embeddings

In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
from langchain_community.vectorstores import Chroma
vector_store = Chroma.from_documents(chunks, embedding)
print("ChromaDB vector store created successfully from documents.")

ChromaDB vector store created successfully from documents.


## Retrieval Setup

In [ ]:
!pip install rank_bm25

In [ ]:
from langchain_community.retrievers import BM25Retriever

# Initialize BM25 retriever from documents
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4 # Retrieve top 4 documents (increased from 2)

# Initialize vector store retriever
vectorstore_retriever = vector_store.as_retriever(search_kwargs={"k": 4}) # Retrieve top 4 documents (increased from 2)

# Custom hybrid retrieval function to combine results from BM25 and VectorStore
def hybrid_retriever_invoke(query):
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = vectorstore_retriever.invoke(query)

    # Combine and de-duplicate documents
    combined_docs = {}
    for doc in bm25_docs + vector_docs:
        combined_docs[doc.page_content] = doc

    return list(combined_docs.values())

print("BM25 and VectorStore retrievers initialized. Custom hybrid retrieval function created.")

BM25 and VectorStore retrievers initialized. Custom hybrid retrieval function created.


In [ ]:
# Perform a sample query using the custom hybrid retriever
query = "What are the main issues with the phone?"
retrieved_docs = hybrid_retriever_invoke(query)

print("Query:", query)
print("\nRetrieved documents:")
for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1}:\n{doc.page_content}\n---")

Query: What are the main issues with the phone?

Retrieved documents:
Document 1:
No issues till now You can buy it for students daily use or for your parents who are started to use Totally worth on this price 6 people found this helpful
---
Document 2:
The phone often lags when opening multiple apps. Disappointed with the performance.
---
Document 3:
According to price , it is the best . Screen is large , speakers are loud , long lasting battery,light weight and usable camera. Best for giving to grandparents who want to learn using smartphone
---
Document 4:
Biggest problem with the phone is, if you go out of network, it won't auto register until you move it to flight mode and remove once
---


## Cross Encoder Reranker

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

# Load a pre-trained Cross-Encoder model
# This model is specifically trained for re-ranking tasks
reranker_model = CrossEncoder('cross-encoder/ms-marco-TinyBERT-L-2')

print("Cross-Encoder reranker model loaded successfully.")

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-TinyBERT-L-2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-Encoder reranker model loaded successfully.


In [ ]:
def rerank_documents(query, documents, model):
    # Prepare sentences for the cross-encoder: (query, document_text) pairs
    sentence_pairs = [[query, doc.page_content] for doc in documents]

    # Get scores from the cross-encoder
    # The scores indicate the relevance of each document to the query
    scores = model.predict(sentence_pairs)

    # Combine documents with their scores and sort them in descending order of score
    ranked_docs_with_scores = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)

    # Return only the re-ranked documents
    return [doc for doc, score in ranked_docs_with_scores]

# Rerank the previously retrieved documents
query = "What are the main issues with the phone?" # Re-define query for consistency with prior reranking test
retrieved_docs = hybrid_retriever_invoke(query) # Re-retrieve docs with potentially new data
reranked_docs = rerank_documents(query, retrieved_docs, reranker_model)

print("Reranked documents:")
for i, doc in enumerate(reranked_docs):
    print(f"Document {i+1} (Score: {reranker_model.predict([query, doc.page_content]):.4f}):\n{doc.page_content}\n---")

Reranked documents:
Document 1 (Score: 0.1479):
Biggest problem with the phone is, if you go out of network, it won't auto register until you move it to flight mode and remove once
---
Document 2 (Score: 0.0028):
According to price , it is the best . Screen is large , speakers are loud , long lasting battery,light weight and usable camera. Best for giving to grandparents who want to learn using smartphone
---
Document 3 (Score: 0.0024):
The phone often lags when opening multiple apps. Disappointed with the performance.
---
Document 4 (Score: 0.0012):
No issues till now You can buy it for students daily use or for your parents who are started to use Totally worth on this price 6 people found this helpful
---


## LLM Integration and RAG Chain

In [ ]:
!pip install langchain-google-genai langchain

In [ ]:
!pip install langchain_groq

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_groq import ChatGroq
import os
from getpass import getpass

# Initialize the LLM with Groq (moved here for consistent usage)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Please provide your Groq API key: ")
llm = ChatGroq(model="openai/gpt-oss-120b", groq_api_key=os.environ["GROQ_API_KEY"])

# Define a function to format documents for the prompt, including metadata
def format_docs(docs):
    formatted_strings = []
    for doc in docs:
        content = doc.page_content
        metadata_str = ""
        if doc.metadata:
            # Include sentiment and extracted issues if present
            meta_items = []
            if 'sentiment_label' in doc.metadata:
                meta_items.append(f"Sentiment: {doc.metadata['sentiment_label']}")
            if 'extracted_issues' in doc.metadata and doc.metadata['extracted_issues']:
                issues_str = "; ".join(doc.metadata['extracted_issues'])
                meta_items.append(f"Identified Issues: {issues_str}")
            if meta_items:
                metadata_str = f" [Metadata: {', '.join(meta_items)}]\n"
        formatted_strings.append(f"{content}{metadata_str}")
    return "\n\n".join(formatted_strings)

print("LLM initialized with Groq and format_docs function defined.")

LLM initialized with Groq and format_docs function defined.


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

# Define Pydantic model for structured output
class PhoneReviewAnalysis(BaseModel):
    main_issues: List[str] = Field(description="List of negative issues or problems identified in the phone review.")
    good_features: List[str] = Field(description="List of positive features or strengths of the phone.")

# Set up the parser
parser = PydanticOutputParser(pydantic_object=PhoneReviewAnalysis)

# Define the prompt template with format instructions
rag_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Based on the following context, extract the phone's main issues and good features. Your output must be a JSON object with two fields: 'main_issues' (a list of distinct and concise negative aspects) and 'good_features' (a list of distinct and concise positive aspects). Use the exact JSON structure provided by the format instructions. If no information is found for a category, return an empty list for that category.\n{format_instructions}\nContext:\n{context}"),
    ("user", "{question}"),
])

# Re-create the RAG chain to use the new LLM and parser
rag_chain = (
    {
        "context": lambda x: format_docs(rerank_documents(x["question"], hybrid_retriever_invoke(x["question"]), reranker_model)),
        "question": RunnablePassthrough(),
        "format_instructions": lambda x: parser.get_format_instructions()
    }
    | rag_prompt_template
    | llm
    | parser
)

print("RAG chain re-created to use Groq LLM and PydanticOutputParser for JSON output.")

RAG chain re-created to use Groq LLM and PydanticOutputParser for JSON output.


In [ ]:
# Test the RAG chain with a query
final_query = "What are the main issues reported with this phone and what are its good features?"

print(f"Final Query: {final_query}")
print("\nGenerated Answer:")
print(rag_chain.invoke({"question": final_query}))

Final Query: What are the main issues reported with this phone and what are its good features?

Generated Answer:
main_issues=['Does not auto‑register when out of network, requiring flight‑mode toggle to reconnect'] good_features=['Great build quality', 'Snappy performance for its price', 'Surprisingly good gaming performance', 'Large display', 'Loud speakers', 'Long‑lasting battery', 'Lightweight design', 'Usable camera', 'Easy to use for seniors and beginners', 'Excellent value for price']


## Evaluate RAG Chain Accuracy

In [ ]:
import json
from typing import Dict, Any

# Define a sample test dataset with expected structured outputs
test_dataset = [
    {
        "query": "What are the main problems and advantages of this phone?",
        "expected_output": PhoneReviewAnalysis(
            main_issues=[
                "Unreliable Wi\u2011Fi connection that drops and switches to mobile data",
                "Network auto\u2011registration fails when out of coverage unless flight mode is toggled",
                "Camera and gaming performance are weak, as expected from an entry\u2011level device",
                "Battery life shorter than advertised",
                "Phone lags when opening multiple apps",
                "Battery drain after the last update",
                "Camera is terrible in low light"
            ],
            good_features=[
                "Large 6.7\u2011inch display",
                "Big battery capacity",
                "Runs Android 16 with One UI 8",
                "Handles everyday tasks (browsing, WhatsApp, etc.) well",
                "Mid\u2011range feel and build quality despite budget price",
                "Ideal for students or parents for daily use",
                "Product arrived quickly",
                "Great display and camera for a budget phone",
                "Underrated, great build quality",
                "Snappy performance for its price point",
                "Gaming is surprisingly good",
                "New UI is super smooth and responsive"
            ]
        )
    },
    {
        "query": "What are the drawbacks of the phone described in the reviews?",
        "expected_output": PhoneReviewAnalysis(
            main_issues=[
                "Dropped Wi\u2011Fi connection despite close proximity to the router",
                "Automatic switching to mobile data which can consume a lot of data",
                "Phone malfunctions from the beginning despite no physical damage",
                "Phone frequently switches off by itself",
                "Wi\u2011Fi connectivity problems",
                "Bluetooth connectivity problems",
                "Poor customer care with inadequate support",
                "Network auto\u2011registration fails when out of coverage unless flight mode is toggled",
                "Inability to handle heavy gaming (1GB+ games) causing it to hang",
                "Battery life much shorter than advertised",
                "Phone often lags when opening multiple apps",
                "Battery drain after the last update",
                "Camera is a joke / blurry photos in anything less than perfect lighting",
                "Camera is still terrible in low light",
                "Durability concerns (shattered after one drop)"
            ],
            good_features=[]
        )
    },
    {
        "query": "What are the positive aspects of the phone?",
        "expected_output": PhoneReviewAnalysis(
            main_issues=[],
            good_features=[
                "Large screen",
                "Loud speakers",
                "Long lasting battery",
                "Light weight",
                "Usable camera",
                "Good entry level phone",
                "Large 6.7 in display",
                "Android 16 with One UI 8",
                "Great for normal use",
                "Quality and feel of a midrange product",
                "Does all essentials well (Internet browsing, WhatsApp)",
                "Good for students daily use",
                "Good for parents who are starting to use smartphones",
                "Product arrived quickly",
                "Great display",
                "Highly recommend for casual users",
                "Great build quality",
                "Snappy performance for its price point",
                "Gaming is surprisingly good",
                "New UI is super smooth and responsive"
            ]
        )
    }
]

def calculate_f1_score(predicted_list: List[str], gold_list: List[str]) -> float:
    if not gold_list and not predicted_list:
        return 1.0 # Perfect score if both are empty
    if not gold_list or not predicted_list:
        return 0.0 # Zero score if one is empty but not the other

    # Normalize and lower case for case-insensitive comparison
    predicted_set = set(item.lower().strip() for item in predicted_list)
    gold_set = set(item.lower().strip() for item in gold_list)

    true_positives = len(predicted_set.intersection(gold_set))
    false_positives = len(predicted_set - gold_set)
    false_negatives = len(gold_set - predicted_set)

    if (true_positives + false_positives) == 0:
        precision = 0.0
    else:
        precision = true_positives / (true_positives + false_positives)

    if (true_positives + false_negatives) == 0:
        recall = 0.0
    else:
        recall = true_positives / (true_positives + false_negatives)

    if (precision + recall) == 0:
        f1 = 0.0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
    return f1

def evaluate_rag_chain(dataset: List[Dict[str, Any]], chain) -> Dict[str, Any]:
    total_issue_f1 = 0.0
    total_feature_f1 = 0.0
    num_tests = len(dataset)

    print(f"\n--- Evaluating RAG Chain on {num_tests} Samples ---\n")

    for i, test_case in enumerate(dataset):
        query = test_case["query"]
        expected_output = test_case["expected_output"]

        print(f"Test Case {i+1}:")
        print(f"  Query: {query}")

        try:
            generated_output = chain.invoke({"question": query})
            print(f"  Generated Issues: {generated_output.main_issues}")
            print(f"  Expected Issues: {expected_output.main_issues}")
            print(f"  Generated Features: {generated_output.good_features}")
            print(f"  Expected Features: {expected_output.good_features}")

            issue_f1 = calculate_f1_score(generated_output.main_issues, expected_output.main_issues)
            feature_f1 = calculate_f1_score(generated_output.good_features, expected_output.good_features)

            total_issue_f1 += issue_f1
            total_feature_f1 += feature_f1

            print(f"  Issue F1 Score: {issue_f1:.4f}")
            print(f"  Feature F1 Score: {feature_f1:.4f}")
            print("----------------------------------------")

        except Exception as e:
            print(f"  Error during chain invocation: {e}")
            print("----------------------------------------")
            num_tests -= 1 # Reduce count for tests that errored out

    if num_tests == 0:
        return {"average_issue_f1": 0.0, "average_feature_f1": 0.0}

    average_issue_f1 = total_issue_f1 / num_tests
    average_feature_f1 = total_feature_f1 / num_tests

    print(f"\n--- Evaluation Summary ---")
    print(f"Average Issue F1 Score: {average_issue_f1:.4f}")
    print(f"Average Feature F1 Score: {average_feature_f1:.4f}")

    return {
        "average_issue_f1": average_issue_f1,
        "average_feature_f1": average_feature_f1,
    }

# Update test_dataset with all collected reviews if necessary
# Assuming test_dataset is already defined in a previous cell

# Run the evaluation
evaluation_results = evaluate_rag_chain(test_dataset, rag_chain)
print(f"\nEvaluation Results: {json.dumps(evaluation_results, indent=2)}")


--- Evaluating RAG Chain on 3 Samples ---

Test Case 1:
  Query: What are the main problems and advantages of this phone?
  Generated Issues: ['Dropped Wi‑Fi connection even when near the router', 'Automatic switching to mobile data, risking unintended data usage', 'Phone fails to auto‑register on network when out of coverage, requiring flight‑mode toggle']
  Expected Issues: ['Unreliable Wi‑Fi connection that drops and switches to mobile data', 'Network auto‑registration fails when out of coverage unless flight mode is toggled', 'Camera and gaming performance are weak, as expected from an entry‑level device', 'Battery life shorter than advertised', 'Phone lags when opening multiple apps', 'Battery drain after the last update', 'Camera is terrible in low light']
  Generated Features: ['Budget price with mid‑range build quality', 'Handles essential apps (browsing, WhatsApp) well', 'Large display', 'Loud speakers', 'Long‑lasting battery', 'Lightweight design', 'Usable camera', 'Suitable